# **Reproduction and Extensions of**
## *An Analysis of Model-Based Interval Estimation for Markov Decision Processes*

**Course:** Seminar Advanced Topics in Reinforcement Learning  
**Professor:** Christos Dimitrakakis  
**Supervisor:** Victor Villin  
**Students:** Allizha Theiventhiram, Rithika Shyam Kumar, Aurélie Wasem, Boris Verdecia Echarte  

---

## **Paper Reference**

**Strehl, A. L., & Littman, M. L. (2008).**  
>[*An Analysis of Model-Based Interval Estimation for Markov Decision Processes*](https://www.sciencedirect.com/science/article/pii/S0022000008000767?ref=pdf_download&fr=RR-2&rr=9e42619f7ebf6aa0#aep-abstract-id7)

Journal of Computer and System Sciences, 74(8), pages 1309–1331.  

### **Authors**

- **Alexander L. Strehl** — Yahoo! Inc., Sunnyvale, CA, USA  
- **Michael L. Littman** — Computer Science Department, Rutgers University, NJ, USA  

---
## **Objectives**
>- understand the theoretical foundations of optimism-based exploration in MDPs,
>- implement and compare MBIE, MBIE-EB, and baseline algorithms such as **E3** and **R-max**,
>-  reproduce benchmark experiments (e.g., RiverSwim and SixArms),
>- and explore possible extensions, including alternative exploration strategies or more challenging environments.

This work aims to provide both a theoretical and empirical understanding of efficient exploration in reinforcement learning.

---
# **Part 1: Theoretical Foundations**
---

## **1. Introduction**

In reinforcement learning (RL), an agent interacts with an environment over time and tries to maximize the total reward it receives. Unlike supervised learning, the agent is **not given a dataset in advance**. Instead, it must learn by acting, observing the consequences of its actions, and updating its behavior accordingly.

When the environment is unknown, the agent faces the fundamental **exploration–exploitation dilemma**:

- **Exploitation:** choose actions that seem best according to current knowledge  
- **Exploration:** choose actions that may be uncertain, in order to gain information and improve future decisions  

This trade-off is at the heart of RL. If the agent explores too little, it may never discover better actions. If it explores too much, it wastes time on suboptimal behavior.

A simple strategy such as $\varepsilon$-greedy exploration chooses a random action with some small probability $\varepsilon$. While easy to implement, this approach does **not use uncertainty in a structured way**: it explores randomly rather than focusing on actions that are uncertain but potentially valuable. As a result, it can be inefficient, especially in large or hard environments.

**Key question addressed in this work:**

> How can we explore efficiently by focusing on *uncertainty*, rather than exploring randomly?

The central idea is to replace random exploration with **directed exploration**, where the agent actively prefers actions that are both uncertain and potentially optimal.

The paper studies this problem in the setting of **finite Markov Decision Processes (MDPs)**, where the reward and transition dynamics are initially unknown. Its goal is to design algorithms that learn efficiently and behave near-optimally with high probability.

---

## **1.1 Interval Estimation (IE) and Optimistic Exploration**

The main intuition behind the paper comes from **Interval Estimation (IE)**, a method originally developed for the simpler **multi-armed bandit** setting.

### **1.1.1 Bandit setting**

In a $k$-armed bandit problem, the agent repeatedly chooses one of $k$ possible actions, called *arms*. Each arm produces a reward drawn from an unknown probability distribution with mean $\mu_i$.

The objective is to identify the arm with the highest expected reward:
$$
\arg\max_i \mu_i.
$$

The difficulty is that the values $\mu_i$ are unknown, so the agent must estimate them through repeated interaction.

This setting is simpler than an MDP because:
- each action gives an immediate reward,
- there is no notion of state,
- actions do not influence future transitions.

Still, it captures the core exploration problem in a minimal form.

---

### **1.1.2 General form of Interval Estimation**

For each arm $i$, the agent keeps:
- an empirical estimate $\hat{\mu}_i$ of the mean reward,
- a confidence interval:
$$
[\hat{\mu}_i - \epsilon_i,\; \hat{\mu}_i + \epsilon_i]
$$

Here:
- $\hat{\mu}_i$ tells us how good the arm appears based on observed data,
- $\epsilon_i$ measures how uncertain this estimate still is.

The IE strategy then selects the arm with the highest **upper confidence bound**:
$$
a_t = \arg\max_i \left( \hat{\mu}_i + \epsilon_i \right).
$$

This is the key decision rule.

**Example (intuition)**

Suppose we have two arms:

- Arm 1: $\hat{\mu}_1 = 0.8$, $\epsilon_1 = 0.05$ → upper bound = 0.85  
- Arm 2: $\hat{\mu}_2 = 0.6$, $\epsilon_2 = 0.3$ → upper bound = 0.9  

Even though Arm 2 has a lower estimated reward, it is selected because it is more uncertain.

This illustrates how IE naturally favors **uncertain but promising actions**.

---

### **1.1.3 Key intuition: optimism**

Why does this work?

- If an arm has been sampled many times, then its estimate is reliable, so $\epsilon_i$ becomes small.
- If an arm has been sampled only a few times, then the uncertainty is large, so $\epsilon_i$ remains large.

As a result:
- **well-known actions** are chosen because they have good estimated rewards,
- **uncertain actions** are chosen because they might still turn out to be optimal.

This is exactly the principle of **optimism in the face of uncertainty**:

> When the agent is unsure, it behaves as if the uncertain option could be very good.

This idea is powerful because it unifies exploration and exploitation in a single rule. The algorithm does not need a separate “exploration mode”: uncertainty itself drives exploration.

---

### **1.1.4 From Bandits to MDPs**

The paper extends this idea from bandits to full MDPs. This is much harder, because in an MDP:

- actions influence **future states**, not only immediate rewards,
- uncertainty affects both **rewards** and **transitions**,
- the quality of an action depends on its **long-term consequences**.

In a bandit, the question is simply: *Which arm has the highest mean reward?*  
In an MDP, the question becomes: *Which action leads to the highest total discounted return, taking into account future transitions and rewards?*

Therefore, instead of being optimistic about a single number (the mean reward), we must be optimistic about an entire **dynamical system** (the MDP). This requires reasoning about how uncertainty propagates through future states and rewards.

---

## **1.2 Markov Decision Processes (MDPs)**

The environment is modeled as a finite discounted MDP:
$$
M = \langle S, A, T, R, \gamma \rangle
$$

where:

- $S$ is the finite set of states,
- $A$ is the finite set of actions,
- $T(s' \mid s,a)$ is the probability of moving to state $s'$ after taking action $a$ in state $s$,
- $R(s,a)$ is the expected immediate reward obtained by taking action $a$ in state $s$,
- $\gamma \in [0,1)$ is the discount factor.

The discount factor controls the importance of future rewards:
- if $\gamma$ is close to $0$, the agent focuses mostly on immediate rewards,
- if $\gamma$ is close to $1$, future rewards matter much more.

---

### **1.2.1 Interaction model**

At each time step:

1. the agent observes the current state $s$,
2. chooses an action $a$,
3. receives a reward $r \sim R(s,a)$,
4. transitions to a next state $s' \sim T(s,a)$.

This produces a trajectory of the form:
$$
s_1, a_1, r_1, s_2, a_2, r_2, \dots
$$

The agent does not know the true model in advance, so it must learn from these interactions.

---

### **1.2.2 Learning setting**

The learner is assumed to know:
- the state space $S$,
- the action space $A$,
- the discount factor $\gamma$,
- and the accuracy and confidence parameters $\varepsilon$ and $\delta$.

However, it does **not know**:
- the transition probabilities $T$,
- the reward function $R$.

Its objective is to learn a policy that is **near-optimal**, meaning that its value is within $\varepsilon$ of the optimal one, with high probability.

---

## **1.3 Policies and Value Functions**

A **policy** $\pi$ specifies how the agent chooses actions. In the simplest case, a policy maps each state to an action.

To evaluate a policy, we use value functions.

---

### **1.3.1 Value functions**

The **state value function** of a policy $\pi$ is:
$$
V^\pi(s) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \right]
$$

It represents the expected total discounted reward obtained by starting from state $s$ and following policy $\pi$.

The **action value function** is:
$$
Q^\pi(s,a) = \mathbb{E}\left[r_0 + \gamma V^\pi(s_1)\right]
$$

It represents the expected discounted return obtained by first taking action $a$ in state $s$, and then following policy $\pi$ afterward.

So:

- $V^\pi(s)$ tells us how good a state is under policy $\pi$,
- $Q^\pi(s,a)$ tells us how good a specific action is in a given state.

**Intuition**

- $V^\pi(s)$ answers: *“How good is it to be in state $s$?”*  
- $Q^\pi(s,a)$ answers: *“How good is it to take action $a$ in state $s$?”*

---

### **1.3.2 Optimality**

The **optimal value function** is:
$$
V^*(s) = \max_\pi V^\pi(s)
$$

and the corresponding optimal action-value function is:
$$
Q^*(s,a).
$$

A policy is considered **$\varepsilon$-optimal** if its value is close to optimal:
$$
V^\pi(s) \geq V^*(s) - \varepsilon
$$

for all relevant states $s$.

The overall goal of RL is therefore to learn a policy that is as close as possible to the optimal policy.

---

## **1.4 Optimism in Model-Based RL (MBIE Principle)**

In the MDP setting, optimism is applied not only to rewards, but to the **entire estimated model of the environment** (both rewards and transitions).

**Key idea**

Instead of trusting the empirical model, the agent considers all models that are plausible given the data, and chooses the one that yields the highest possible value.

A simplified optimistic action-value estimate has the form:
$$
Q(s,a) = \hat{R}(s,a) + \gamma \sum_{s'} \hat{T}(s' \mid s,a)\, V(s') + b(s,a)
$$

where:
- $\hat{R}(s,a)$ is the empirical reward estimate,
- $\hat{T}(s' \mid s,a)$ is the empirical transition estimate,
- $b(s,a)$ is an exploration bonus.

The exploration bonus is large when $(s,a)$ has been visited only a few times, and small when it is well known.

---

### **Intuition**

This means:

- if a state–action pair is well explored, then its estimated value mostly reflects exploitation,
- if it is poorly explored, then the bonus increases its value and encourages exploration.

So the algorithm behaves as if **uncertain parts of the environment might be especially good**. This makes exploration directed rather than random.

This is the essential idea behind **MBIE** and **MBIE-EB**.

---

## **1.5 Overview of Main Results**

The paper introduces two algorithms:

- **MBIE (Model-Based Interval Estimation)**
- **MBIE-EB (MBIE with Exploration Bonus)**

Both are designed to solve the exploration problem in unknown MDPs using optimism.

The main theoretical result is that these algorithms are **PAC-MDP**, which means that, with high probability:

- they behave near-optimally on all but a **polynomial number of time steps**,
- and their computational complexity is also polynomial in the relevant problem parameters.

The paper also introduces a new performance metric called **average loss**, which measures how much reward the agent actually loses during learning. This complements the PAC-MDP analysis, which focuses on how often the agent behaves suboptimally.

---

## **2. Performance Metrics**

To claim that an RL algorithm is efficient, we need a precise definition of what “efficient” means.

A natural idea is that a good algorithm should achieve **near-optimal performance with high probability**. This is the spirit of the **Probably Approximately Correct (PAC)** framework.

However, reinforcement learning is more subtle than supervised learning because:

> the agent’s actions affect which data it observes.

This means that learning and evaluation cannot be separated so easily. A poor decision early in training may influence the entire future trajectory.

---

## **2.1 Challenges in RL Evaluation**

The main difficulty is that **mistakes made early can affect all later rewards**.

For example, imagine that a poor action at the beginning sends the agent into a part of the state space with very low rewards. Because rewards are discounted, even if the agent later improves, it may never fully recover from this early mistake.

So a meaningful evaluation metric should:
- account for sequential decision-making,
- reflect the fact that learning happens online,
- and not unfairly penalize unavoidable early uncertainty.

---

## **2.2 Previous PAC Approaches**

### **2.2.1 Reset-based framework**

An early PAC-style approach assumes that the agent learns through repeated trials from a fixed start state.

This makes evaluation simpler:
- each trial begins from the same state,
- the agent can be judged based on performance from that state.

This framework is attractive because it resembles classical PAC learning. However, it is also restrictive: many RL problems do **not** allow resets.

---

### **2.2.2 E3 framework**

A more general approach, used in the E3 framework, removes the reset assumption.

However, this raises a new question:  
**From which state should optimality be measured?**

Possible answers include:

- the initial state,
- the average over visited states,
- the final state reached by the learner.

Each choice has drawbacks. In particular:
- evaluating from the initial state is too strict, because early mistakes are unavoidable,
- averaging across states may ignore the role of discounting.

The E3 analysis therefore evaluates performance from the **final state reached**. This is more general than reset-based learning, but still separates learning from evaluation.

---

## **2.3 Online Evaluation: Sample Complexity**

The paper adopts a more natural online viewpoint: evaluate the **learning algorithm itself** while it is running.

Let:
- $A_t$ be the policy followed by the algorithm at time $t$,
- $s_t$ be the current state.

---

### **Definition 1 (Sample Complexity of Exploration)**

The **sample complexity of exploration** is the number of time steps $t$ such that:
$$
V^{A_t}(s_t) < V^*(s_t) - \varepsilon
$$

In words, it counts the number of times the agent is **not $\varepsilon$-optimal** from the state it is currently in.

---

### **Why this matters**

This metric is important because it measures performance **during learning**, not only after learning.

It also avoids requiring the algorithm to explicitly announce:  
> “I am now optimal.”

Instead, it only requires that the algorithm behaves well **most of the time**.

---

## **2.4 PAC-MDP Framework**

Using sample complexity, we can now define efficient learning in MDPs.

---

### **Definition 2 (PAC-MDP Algorithm)**

An algorithm $A$ is **PAC-MDP** if, for any $\varepsilon > 0$ and $\delta > 0$:

- its per-step computational complexity is polynomial,
- its sample complexity is bounded by a polynomial,

in the quantities:
$$
|S|,\; |A|,\; \frac{1}{\varepsilon},\; \frac{1}{\delta},\; \frac{1}{1-\gamma}
$$

with probability at least $1 - \delta$.

---

### **Interpretation**

This means:
- the algorithm may behave suboptimally sometimes,
- but the number of such time steps is controlled,
- and both learning time and computation remain tractable.

This is different from classical PAC learning, where errors are often assumed to happen before the final hypothesis is produced. In RL, mistakes may occur later, because some parts of the state space are only discovered late.

---

## **2.5 Limitation of Sample Complexity**

Sample complexity is very useful theoretically, but it has an important limitation:

it is defined in terms of **expected value**, not the actual rewards collected by the agent.

So it does not directly answer:
> How much reward did the agent really lose while learning?

To address this, the paper introduces another metric.

---

## **2.6 Average Loss**

The goal of **average loss** is to measure how far the agent’s actual reward trajectory is from optimal behavior.

---

### **Definition 3 (Instantaneous Loss and Average Loss)**

Consider a trajectory of length $H$:
$$
s_1, a_1, r_1, \dots, s_H, a_H, r_H
$$

The **instantaneous loss** at time $t$ is:
$$
il(t) = V^*(s_t) - \sum_{i=t}^{H} \gamma^{\,i-t} r_i
$$


This compares:
- the optimal value from state $s_t$,
- the actual discounted reward obtained by the agent from time $t$ onward.

**Intuition**

At each time step $t$, we compare:

- what the agent *could have obtained* (optimal value $V^*(s_t)$),
- what it *actually obtained*.

So the loss measures the **regret due to imperfect decisions during learning**.

The **average loss** is then:
$$
l = \frac{1}{H} \sum_{t=1}^{H} il(t)
$$

---

### **Interpretation**

- If $l$ is small, the agent behaves close to optimally on average.
- If $l$ is large, the agent loses a significant amount of reward due to poor decisions.

So average loss is more directly tied to the agent’s **real online performance** than sample complexity.

---

## **2.7 Relationship Between Metrics**

One of the important conceptual results of the paper is that:

> **Low sample complexity implies low average loss**

This means the PAC-MDP framework is not only mathematically convenient, but also behaviorally meaningful: if an algorithm is suboptimal only a limited number of times, then its total reward performance during learning will also be good.

---

## **3. Model-Based Interval Estimation (MBIE)**

After introducing the exploration problem and the performance metrics, we can now describe the main algorithmic contribution of the paper: **Model-Based Interval Estimation (MBIE)**.

MBIE extends the optimism principle from bandits to MDPs by maintaining uncertainty estimates over the learned model and planning optimistically.

In other words, instead of asking:

> “What is the best action according to my current estimate?”

MBIE asks:

> “What is the best action according to the most optimistic model still consistent with my data?”

This is the key difference.

---

## **3.1 Main Quantities Maintained by MBIE**

For each state–action pair $(s,a)$, MBIE maintains:

- $\tilde{Q}(s,a)$: optimistic estimate of the action value  
- $\hat{R}(s,a)$: empirical estimate of the expected reward  
- $\hat{T}(s' \mid s,a)$: empirical estimate of the transition probabilities  
- $n(s,a)$: number of times $(s,a)$ has been visited  
- $n(s,a,s')$: number of times $(s,a)$ led to state $s'$  

These quantities summarize the agent’s current knowledge of the environment.

---

## **3.2 Empirical Model**

### **Reward estimate**
$$
\hat{R}(s,a) = \frac{1}{n(s,a)} \sum_{i=1}^{n(s,a)} r_i
$$

This is simply the empirical mean of the observed rewards.

### **Transition estimate**
$$
\hat{T}(s' \mid s,a) = \frac{n(s,a,s')}{n(s,a)}
$$

This is the empirical frequency with which the transition $(s,a) \to s'$ has been observed.

Together, $\hat{R}$ and $\hat{T}$ define the current learned model of the MDP.

---

## **3.3 Confidence Intervals**

Because the model is learned from data, these estimates are uncertain. MBIE therefore builds confidence intervals around them.

### **Reward confidence interval**
$$
\hat{R}(s,a) \pm \epsilon_R(n(s,a)), \quad
\epsilon_R(n(s,a)) = \sqrt{\frac{\ln(2/\delta_R)}{2\,n(s,a)}}
$$

### **Transition confidence interval**
$$
\left\| \tilde{T}(s,a) - \hat{T}(s,a) \right\|_1 \leq \epsilon_T(n(s,a))
$$

with
$$
\epsilon_T(n(s,a)) =
\sqrt{\frac{2\left[\ln(2|S| - 2) - \ln(\delta_T)\right]}{n(s,a)}}
$$

These intervals define the set of **plausible models** consistent with the observed data.

---

## **3.4 Optimistic Model Construction**

MBIE chooses the model in this confidence set that gives the highest possible value.

This leads to the optimistic Bellman equation:
$$
\tilde{Q}(s,a)
=
\max_{\tilde{R}, \tilde{T}}
\left[
\tilde{R}(s,a)
+
\gamma \sum_{s'} \tilde{T}(s' \mid s,a)\max_{a'} \tilde{Q}(s',a')
\right]
$$

This equation says:

- use optimistic rewards,
- use optimistic transitions,
- and plan according to the most favorable plausible future.

For an unvisited state–action pair, the algorithm uses optimistic initialization:
$$
\tilde{Q}(s,a) = \frac{1}{1 - \gamma}
$$

which is the maximum possible discounted return.

This guarantees that unknown actions remain attractive until the agent has gathered enough information.

---

## **3.5 Action Selection**

At each time step, the agent chooses the greedy action:
$$
a_t = \arg\max_a \tilde{Q}(s_t,a)
$$

This produces the desired balance:

- actions that look good under the current model are exploited,
- actions that remain uncertain are explored because their optimistic value is still high.

So exploration is not random: it is driven by uncertainty in a purposeful way.

---

## **3.6 Limited Model Updates**

The algorithm introduces a parameter $m$:

- only the first $m$ observations of each $(s,a)$ are used for learning,
- once $n(s,a) = m$, the pair is considered “known.”

This serves two purposes:

1. it limits the number of times the model changes,
2. it makes the theoretical analysis easier, because the policy cannot change infinitely often.

---

## **3.7 MBIE-EB: Exploration Bonus Variant**

The paper also introduces a simpler variant called **MBIE-EB**.

Instead of explicitly optimizing over all plausible MDPs, MBIE-EB adds a bonus directly to the empirical Bellman equation:
$$
\tilde{Q}(s,a)
=
\hat{R}(s,a)
+
\gamma \sum_{s'} \hat{T}(s' \mid s,a)\max_{a'} \tilde{Q}(s',a')
+
\frac{\beta}{\sqrt{n(s,a)}}
$$

This bonus is large for poorly explored state–action pairs and small for well-explored ones.

---

### **Why MBIE-EB is useful**

It keeps the same exploration principle as MBIE, but is:

- simpler to understand,
- easier to implement,
- less computationally demanding.

That is why MBIE-EB is often more practical, even though MBIE is theoretically more explicit in its use of confidence sets.

---

## **3.8 MBIE vs MBIE-EB**

### **MBIE**
- explicit confidence intervals on rewards and transitions,
- optimistic planning over a confidence set of MDPs,
- stronger conceptual connection to Interval Estimation,
- more computationally expensive.

### **MBIE-EB**
- uses the empirical model plus an exploration bonus,
- much simpler computationally,
- easier to implement in practice,
- still enjoys PAC-MDP guarantees.

---

## **3.9 High-Level Algorithm**

A high-level view of MBIE is:

1. Observe the current state $s_t$  
2. Update empirical estimates $\hat{R}$, $\hat{T}$, and visit counts  
3. Build optimistic value estimates $\tilde{Q}$  
4. Solve the optimistic model  
5. Choose the greedy action:
$$
a_t = \arg\max_a \tilde{Q}(s_t,a)
$$
6. Repeat  

---

## **3.10 Main Intuition to Remember**

The most important intuition behind MBIE is:

> **Uncertainty should not lead to random exploration, but to optimistic exploration.**

If a part of the environment is not well understood, MBIE treats it as potentially valuable and explores it systematically. This is what allows the algorithm to combine efficient learning with strong theoretical guarantees.

---

## **Conclusion of Part 1**

We have introduced the theoretical foundations of optimism-based exploration in MDPs, the PAC-MDP framework for evaluating learning efficiency, and the MBIE algorithm, which combines statistical confidence and optimistic planning to achieve efficient exploration.

---
# **Part 3: Theoretical Foundations**
---

## R-Max

In [ ]:
import numpy as np


def value_iteration(n_states, P, R, gamma, tol=0.01):
    """
    Solve an MDP using value iteration.

    Reference: standard planning algorithm, used in R-Max to solve
    the internal optimistic MDP (Section 3, Strehl & Littman 2008).

    Args:
        n_states  : number of states
        P         : transition probabilities, shape (n_states, n_actions, n_states)
        R         : mean rewards, shape (n_states, n_actions)
        gamma     : discount factor
        tol       : convergence threshold — stop when max|V_new - V| < tol

    Returns:
        V      : optimal value vector, shape (n_states,)
        policy : greedy optimal action per state, shape (n_states,)
    """
    V = np.zeros(n_states)

    for _ in range(1000):
        # Q[s,a] = R[s,a] + gamma * sum_{s'} P[s,a,s'] * V[s']
        Q     = R + gamma * np.einsum('ijk,k->ij', P, V)
        V_new = Q.max(axis=1)

        # Check convergence
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new

    # Final Q recomputation to extract the greedy policy
    Q      = R + gamma * np.einsum('ijk,k->ij', P, V)
    policy = Q.argmax(axis=1)
    return V, policy


class RMax:
    """
    R-Max algorithm (Brafman & Tennenholtz, 2002).

    Core idea (Section 6, Strehl & Littman 2008, footnote 12):
        Every state-action pair (s,a) is classified as "known" or "unknown".
        - Unknown (fewer than m visits): fictitious reward = R_max and
          fictitious transition = stay in s. This makes unknown pairs
          look maximally attractive, naturally driving exploration.
        - Known (at least m visits): use empirical estimates of R and T
          built from observed experience.

    At each update, R-Max solves its internal optimistic MDP via value
    iteration and follows the resulting optimal policy.

    Args:
        n_states  : number of states in the MDP
        n_actions : number of actions in the MDP
        gamma     : discount factor (between 0 and 1)
        m         : visit threshold to declare (s,a) "known"
                    (m=16 for RiverSwim, m=6 for SixArms — Section 6)
        R_max     : fictitious reward for unknown pairs,
                    must be >= maximum possible reward in the true MDP
    """

    def __init__(self, n_states, n_actions, gamma, m, R_max):
        # Hyperparameters
        self.m         = m
        self.n_states  = n_states
        self.n_actions = n_actions
        self.gamma     = gamma
        self.R_max     = R_max

        # Current policy: action 0 everywhere by default
        # (updated as soon as a (s,a) pair becomes known)
        self.policy = np.zeros(n_states, dtype=int)

        # Visit counter per (s,a) pair — Section 3, "occupancy counts"
        self.N = np.zeros((n_states, n_actions))

        # Transition counter (s,a) -> s' — Section 3, "next-state counts"
        self.N_sas = np.zeros((n_states, n_actions, n_states))

        # Cumulative reward sum for (s,a)
        # used to compute R_hat(s,a) = R_sum / N
        self.R_sum = np.zeros((n_states, n_actions))

    def _is_known(self, s, a):
        """
        Return True if (s,a) has been visited at least m times.

        Direct definition from Section 4.3 (page 1317):
        "the set of known state-action pairs K is defined to be
        those (s,a) experienced at least m times"
        """
        return self.N[s, a] >= self.m

    def _build_optimistic_model(self):
        """
        Build the internal optimistic MDP used by R-Max.

        For each (s,a):
            - If known   : use empirical estimates R_hat(s,a) and T_hat(s'|s,a)
            - If unknown : fictitious reward R_max + fictitious transition
                           (stay in s), making this pair maximally attractive

        This construction is the core of R-Max — it forces the agent
        to explore unknown regions before exploiting known ones.

        Returns:
            P_model : internal MDP transitions, shape (S, A, S)
            R_model : internal MDP rewards, shape (S, A)
        """
        S, A    = self.n_states, self.n_actions
        P_model = np.zeros((S, A, S))
        R_model = np.zeros((S, A))

        for s in range(S):
            for a in range(A):
                if self._is_known(s, a):
                    # Empirical mean reward estimate
                    R_model[s, a] = self.R_sum[s, a] / self.N[s, a]
                    # Empirical transition probability estimate
                    P_model[s, a] = self.N_sas[s, a] / self.N[s, a]
                else:
                    # Optimism: fictitious maximum reward
                    R_model[s, a]    = self.R_max
                    # Optimism: fictitious transition — agent stays in s
                    P_model[s, a, s] = 1.0

        return P_model, R_model

    def _update_policy(self):
        """
        Recompute the optimal policy by solving the internal optimistic MDP.

        Only called when a (s,a) pair has just transitioned from "unknown"
        to "known" — this limits the number of costly value iteration calls.
        """
        P_model, R_model = self._build_optimistic_model()
        _, self.policy   = value_iteration(
            self.n_states, self.n_actions,
            P_model, R_model, self.gamma
        )

    def select_action(self, state):
        """
        Return the action to take from the given state.

        The agent simply follows the current policy — which is optimistic
        for unknown states and near-optimal for known ones.

        Args:
            state : current state of the agent

        Returns:
            action : integer between 0 and n_actions-1
        """
        return self.policy[state]

    def update(self, s, a, r, s_next):
        """
        Update the internal model after observing transition (s, a, r, s').

        Following Section 3 (page 1313): only the first m experiences of
        each (s,a) are stored. Beyond m, observations are ignored to
        preserve the theoretical guarantees of the algorithm.

        If (s,a) just reached exactly m visits, the policy is recomputed
        since this pair transitions from "unknown" to "known".

        Args:
            s      : current state
            a      : action taken
            r      : reward received
            s_next : next state observed
        """
        # Ignore visits beyond m (Section 3, "model size limit")
        if self.N[s, a] < self.m:
            self.N[s, a]             += 1
            self.N_sas[s, a, s_next] += 1
            self.R_sum[s, a]         += r

            # If (s,a) just reached m visits → it becomes "known"
            # → recompute the policy
            if self.N[s, a] == self.m:
                self._update_policy()